In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor
import warnings
import xgboost as xgb
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_theme()

In [2]:
train = pd.read_csv("data/track_k_ecommerce_returns_train.csv")
test = pd.read_csv("data/track_k_ecommerce_returns_test.csv")

In [3]:
from sklearn.tree import DecisionTreeRegressor

X_threshold = train['price_eur']
y = train['return_rate_pct']

threshold_finder = DecisionTreeRegressor(max_depth=1, random_state=42)
threshold_finder.fit(X_threshold.values.reshape(-1, 1), y)

optimal_threshold = threshold_finder.tree_.threshold[0]
print(f"The optimal split point is: {optimal_threshold:.2f}")

The optimal split point is: 173.96


We're using DecisionTreeRegressor at 1 depth to make the sole most efficient cut of our data

In [4]:
df_low_price = train[train['price_eur'] <= optimal_threshold]
df_high_price = train[train['price_eur'] > optimal_threshold]

X_low = df_low_price.drop(columns=['return_rate_pct'])
y_low = df_low_price['return_rate_pct']

X_high = df_high_price.drop(columns=['return_rate_pct'])
y_high = df_high_price['return_rate_pct']

Separating the datas into two frames so we can train two models.

In [5]:
#Test on RdForest

model_low_tier = RandomForestRegressor(n_estimators=100, random_state=42)
model_high_tier = RandomForestRegressor(n_estimators=100, random_state=42)

model_low_tier.fit(X_low, y_low)
print("Low_Done")
model_high_tier.fit(X_high, y_high)
print("High_Done")

Low_Done
High_Done


In [8]:
def predict_return_rate(new_data_df, threshold, model_low, model_high):
    #To hold onto the predictions
    predictions = np.zeros(len(new_data_df))
    
    #Sort of boolean list that serve as a filter
    is_low = new_data_df['price_eur'] <= threshold #[True False True ...] True for low tier, False for high tier
    is_high = new_data_df['price_eur'] > threshold #[True False True ...] True for high tier, False for low tier
    
    #Low tier predictions
    if is_low.any():
        #New list with index of the low tiers + their features
        X_new_low = new_data_df.loc[is_low, model_low.feature_names_in_]

        #Predictions[is_low] targets the index of the low tiers from above
            # [True False True ...] => Index 1 and 3
        predictions[is_low] = model_low.predict(X_new_low)
        
    # High tier predictions
    if is_high.any():
        #Same principle as low tier, but for high tier

        X_new_high = new_data_df.loc[is_high, model_high.feature_names_in_]
        predictions[is_high] = model_high.predict(X_new_high)
    
        
    return predictions

final_predictions = predict_return_rate(test, optimal_threshold, model_low_tier, model_high_tier)

#print mse
final_mse = mean_squared_error(test['return_rate_pct'], final_predictions)
print(f"Final MSE on test set: {final_mse:.4f}")


Final MSE on test set: 25.4354


We have a better MSE than previously using RdForest(28.57)